# Challenge 04 - Speaker Recognition (GMM)

Recognition of the speaker (31 people) of each audio track using only the MFCCs. A universal GMM (UBM) is trained on the frames of all speakers and adapted to each speaker (MAP adaptation of the means). Each test track is assigned to the speaker whose GMM gives the highest mean log-likelihood. The predictions are saved in `Es6Predictions.txt`.

## Setup

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import copy
import numpy as np
from scipy.signal import savgol_filter
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# Configuration
db_path = '../../DBs/AudioDigits_ML26_27'
train_file = 'audio_ml2627_train.pkl'
test_file = 'audio_ml2627_test_no_labels.pkl'
result_path = '../Es6Predictions.txt'

# Preprocessing
silence_threshold = 175    # frames with c0 below (track max - threshold) are removed
delta_width = 15

# GMM hyperparameters
n_components = 64
covariance_type = 'diag'
reg_covar = 1e-2
relevance = 32             # MAP relevance factor
seeds = [1, 2, 3, 4, 5]

## Helper Functions

Dataset loading, preprocessing of the MFCCs, training of the UBM and of the adapted speaker GMMs, prediction and export.

Preprocessing of each track: c0 is taken relative to its maximum, deltas and delta-deltas are computed, the silence frames (low c0) are removed and then c0 is dropped. The features are standardized with a scaler fitted on the training frames.

In [2]:
def load_dataset(db_path, filename):
    return np.load(os.path.join(db_path, filename), allow_pickle=True)


def delta(m, order):
    # same as librosa.feature.delta(m.T, width=delta_width, order=order).T
    return savgol_filter(m, delta_width, polyorder=order, deriv=order, axis=0, mode='interp')


def preprocess(mfccs):
    tracks = []
    for m in mfccs:
        m = np.asarray(m, dtype=float).copy()
        m[:, 0] -= m[:, 0].max()
        speech = m[:, 0] > -silence_threshold
        x = np.hstack([m, delta(m, 1), delta(m, 2)])
        if speech.sum() >= 20:
            x = x[speech]
        tracks.append(x[:, 1:])
    return tracks


def standardize(train_tracks, test_tracks):
    scaler = StandardScaler().fit(np.vstack(train_tracks))
    return [scaler.transform(m) for m in train_tracks], [scaler.transform(m) for m in test_tracks]


def map_adapt(ubm, frames):
    gmm = copy.deepcopy(ubm)
    resp = ubm.predict_proba(frames)
    n = resp.sum(axis=0) + 1e-10
    alpha = (n / (n + relevance))[:, None]
    gmm.means_ = alpha * (resp.T @ frames / n[:, None]) + (1 - alpha) * ubm.means_
    return gmm


def train_speaker_models(tracks, speakers, seed):
    ubm = GaussianMixture(n_components=n_components, covariance_type=covariance_type, reg_covar=reg_covar,
                          max_iter=200, random_state=seed).fit(np.vstack(tracks))
    return {s: map_adapt(ubm, np.vstack([m for m, p in zip(tracks, speakers) if p == s])) for s in np.unique(speakers)}


def predict_speaker(train_tracks, train_speakers, tracks):
    scores = 0
    for seed in seeds:
        models = train_speaker_models(train_tracks, train_speakers, seed)
        scores = scores + np.array([[g.score(m) for g in models.values()] for m in tracks])
    return np.array(list(models))[scores.argmax(axis=1)]


def save_predictions(predictions, path):
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set contains sessions 1 and 4 (620 tracks, 20 per speaker), the test set sessions 2 and 3 (620 tracks, without labels). Only the MFCCs are used.

In [3]:
train_data = load_dataset(db_path, train_file)
test_data = load_dataset(db_path, test_file)

train_x = preprocess(train_data['mfccs'])
train_y = np.array(train_data['person_ids'])
train_sessions = np.array(train_data['session_ids'])
test_x = preprocess(test_data['mfccs'])

print('Training tracks: %d, test tracks: %d, speakers: %d' % (len(train_x), len(test_x), len(np.unique(train_y))))
print('Speech frames kept: %.1f%%, features per frame: %d' % (100 * sum(len(m) for m in train_x) / sum(len(m) for m in train_data['mfccs']), train_x[0].shape[1]))

Training tracks: 620, test tracks: 620, speakers: 31
Speech frames kept: 22.7%, features per frame: 59


## Model: GMM-UBM with MAP Adaptation

A `GaussianMixture` with 64 diagonal components is trained on the frames of all speakers (UBM). The GMM of each speaker is the UBM with the means moved towards the speaker's frames, weighted by how many frames each component receives (relevance factor 32). The scores are averaged over 5 UBMs trained with different seeds.

## Model Evaluation

The split is done by session to simulate new recording conditions: train on one session and validate on the other, in both directions.

In [4]:
for train_session, val_session in [(1, 4), (4, 1)]:
    tr = train_sessions == train_session
    val = train_sessions == val_session
    x_tr, x_val = standardize([m for m, t in zip(train_x, tr) if t], [m for m, v in zip(train_x, val) if v])
    val_pred = predict_speaker(x_tr, train_y[tr], x_val)
    print('Train session %d -> validation session %d: accuracy %.4f' % (train_session, val_session, np.mean(val_pred == train_y[val])))

Train session 1 -> validation session 4: accuracy 0.7710
Train session 4 -> validation session 1: accuracy 0.7903


## Prediction and Export

The models are retrained on all training tracks (sessions 1 and 4) and each test track is assigned to the most likely speaker. The predictions are saved in `Es6Predictions.txt`, one speaker id per line.

In [5]:
x_train, x_test = standardize(train_x, test_x)
predictions = predict_speaker(x_train, train_y, x_test)
assert len(predictions) == 620
save_predictions(predictions, result_path)

Predictions successfully saved to ../Es6Predictions.txt
